In [31]:
import pandas as pd
import re
from pathlib import Path
from collections import Counter

pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("../data")
RAW_PATH = DATA_DIR / "raw/lyrics_raw.csv"
OUT_PATH = DATA_DIR / "processed/lyrics_clean.csv"

In [32]:
df = pd.read_csv(RAW_PATH)

df.columns

Index(['artist', 'song_title', 'album', 'release_year', 'lyrics', 'source'], dtype='object')

In [33]:
df[[c for c in df.columns if c != "lyrics"][:3] + ["lyrics"]].sample(5, random_state=7)

,artist,song_title,album,lyrics
22,Streetlight Manifesto,Point / Counterpoint,"{'api_path': '/albums/34804', 'cover_art_url': 'https://images.genius.com/19eb8a440ed814b52f12e468c7a4c7ee.640x640x1...","I've got a gun in my hand, but the gun won't cock\nMy finger's on the trigger, but that trigger seems locked\nAnd I ..."
27,Sublime,Badfish,"{'api_path': '/albums/547549', 'cover_art_url': 'https://images.genius.com/888a662c5f1ed76bbad99cae2960c4ab.640x640x...","(Excuse me)\nShut up\n(Oh my God!)\nHow's it going, dude?\nHey, man, what's up?\n(Tell Todd he can turn the radio ba..."
34,The Slackers,Watch This,"{'api_path': '/albums/194401', 'cover_art_url': 'https://images.genius.com/8486ce46a4af10f5e8bc1264e964c744.1000x100...",I said man you better watch this\nShe's gonna make a take down\nShe gonna make you do right\nShe gonna make you stay...
15,Reel Big Fish,Sell Out,"{'api_path': '/albums/167845', 'cover_art_url': 'https://images.genius.com/f00cc2081675774a0cf1930ad09e4628.500x500x...","""Well, I know you can't work in fast food all your life\nBut don't sign that paper tonight"", she said\nBut it's too ..."
18,Sublime,Santeria,"{'api_path': '/albums/547549', 'cover_art_url': 'https://images.genius.com/888a662c5f1ed76bbad99cae2960c4ab.640x640x...","I don't practice Santeria, I ain't got no crystal ball\nWell, I had a million dollars, but I'd, I'd spend it all\nIf..."


In [34]:
### Cleaning ###
BRACKET_TAG_REGEX = re.compile(r"\[(chorus|verse|bridge|intro|outro|pre[- ]chorus|hook|refrain).*?\]", re.IGNORECASE)

def clean_lyrics(text: str) -> str:
    if pd.isna(text):
        return ""
    
    # normalize newlines
    t = str(text).replace("\r\n", "\n").replace("\r", "\n")

    # remove common section labels, like: [Chorus], [Verse 2], etc...
    t = BRACKET_TAG_REGEX.sub("", t)

    # lowercasse
    t = t.lower()

    # remove URLs if they exist
    t = re.sub(r"https?://\S+|www\.\S+", "", t)

    # keep letters + spaces + newlines; drop punctuation/numbers
    t = re.sub(r"[^a-z\s\n']", " ", t)  # keep apostrophes for don't, i'm

    # normalize spaces within lines
    t = re.sub(r"[ \t]+", " ", t)

    # collapse 3+ blank lines to 2 (keeps verse breaks)
    t = re.sub(r"\n{3,}", "\n\n", t)

    # strip whitespace on each line
    t = "\n".join(line.strip() for line in t.split("\n"))

    # remove leading/trailing blank lines
    t = t.strip()

    return t

In [35]:
df["lyrics_raw"] = df["lyrics"].astype(str)
df["lyrics_clean"] =df["lyrics_raw"].apply(clean_lyrics)

# Show a few before and after samples
sample = df.sample(3, random_state=3)[["artist", "title", "lyrics_raw", "lyrics_clean"]] if "artist" in df.columns and "title" in df.columns else df.sample(3, random_state=3)[["lyrics_raw", "lyrics_clean"]]
sample

,lyrics_raw,lyrics_clean
12,Going around causing disturbances\nDon't you know you'll never win\nYou've got your family running around\nTrying to...,going around causing disturbances\ndon't you know you'll never win\nyou've got your family running around\ntrying to...
40,All I need is a walking cane\nTo guide me on my way\n\nAll I need is a walking cane\nTo guide me on my way\nA hat to...,all i need is a walking cane\nto guide me on my way\n\nall i need is a walking cane\nto guide me on my way\na hat to...
9,"Come on, Eileen!\nCome on, Eileen!\n\nPoor old Johnnie Ray\nSounded sad upon the radio\nHe moved a million hearts in...",come on eileen\ncome on eileen\n\npoor old johnnie ray\nsounded sad upon the radio\nhe moved a million hearts in mon...


In [36]:
# Sanity Checks

def count_lines(text: str) -> int:
    return sum(1 for l in text.split("\n") if l.strip())

def word_count(text: str) -> int:
    return len([w for w in text.split() if w.strip()])

checks = pd.DataFrame({
    "lines_raw": df["lyrics_raw"].apply(count_lines),
    "lines_clean": df["lyrics_clean"].apply(count_lines),
    "word_raw": df["lyrics_raw"].apply(word_count),
    "words_clean": df["lyrics_clean"].apply(word_count),
})

checks.describe()

,lines_raw,lines_clean,word_raw,words_clean
count,52.000000,52.000000,52.000000,52.000000
mean,44.403846,44.403846,292.442308,294.365385
std,15.115580,15.115580,130.896219,131.339708
min,16.000000,16.000000,144.000000,144.000000
25%,33.000000,33.000000,222.250000,222.500000
50%,41.000000,41.000000,272.500000,276.000000
75%,48.000000,48.000000,323.000000,330.750000
max,96.000000,96.000000,1046.000000,1052.000000


In [37]:
# Quick top words check
all_words = " ".join(df["lyrics_clean"]).split()
counts = Counter(all_words)

counts.most_common(30)

[('i', 597),
 ('the', 464),
 ('you', 429),
 ('to', 359),
 ('and', 347),
 ('a', 312),
 ('it', 241),
 ('me', 206),
 ('that', 203),
 ('my', 198),
 ("i'm", 162),
 ('on', 156),
 ('in', 147),
 ('all', 141),
 ('your', 121),
 ('so', 116),
 ('but', 116),
 ('oh', 116),
 ('of', 113),
 ("don't", 112),
 ('no', 110),
 ("it's", 108),
 ('be', 107),
 ('know', 104),
 ('just', 99),
 ('never', 91),
 ("can't", 82),
 ('is', 81),
 ('out', 80),
 ('yeah', 80)]

In [38]:
DATA_DIR.mkdir(exist_ok=True)

# Keep original metadata + both lyrics columns
df.to_csv(OUT_PATH, index=False)
OUT_PATH

PosixPath('../data/processed/lyrics_clean.csv')